# Quy Trình Các Bước Feature Engineering - Dự Đoán Số Lượng Sản Phẩm (Quantity)

Notebook này thực hiện tiền xử lý dữ liệu, làm sạch các giá trị khuyết thiếu theo các quy luật đã phát hiện ở bước EDA, tạo thêm các đặc trưng mới và chuẩn hóa dữ liệu để chuẩn bị cho quá trình huấn luyện mô hình.

**Mục tiêu dự đoán (Target):** `Quantity` (Tiền xử lý và lưu đầy đủ các đặc trưng Price Per Unit và Total Spent để linh hoạt huấn luyện).

## Bước 1. Chia Tách Dữ Liệu (Data Splitting)

Để tránh rò rỉ thông tin (Data Leakage), chúng ta thực hiện chia tập dữ liệu thành hai tập Train (80%) và Test (20%) trước khi thực hiện bất kỳ phương pháp điền khuyết hay tính toán thống kê nào.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os

# Load dữ liệu thô
df = pd.read_csv('../data/raw/retail_store_sales.csv')

# Chia Train/Test theo tỷ lệ 80/20 với random_state cố định
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

print(f'Tập huấn luyện (Train): {train_df.shape}')
print(f'Tập kiểm thử (Test): {test_df.shape}')

Tập huấn luyện (Train): (10060, 11)
Tập kiểm thử (Test): (2515, 11)


## Bước 2. Xử Lý Giá Trị Thiếu (Handling Missing Values)

Áp dụng các quy luật làm sạch dữ liệu tùy chỉnh như đã thảo luận:
1. **Price Per Unit**: Điền bằng `Total Spent / Quantity` cho các dòng bị khuyết (nếu có đủ 2 giá trị còn lại).
2. **Item**: Xây dựng bản đồ `(Category, Price Per Unit) -> Item` trên tập Train và dùng nó để khôi phục tên Item bị thiếu trên cả tập Train và Test.
3. **Xóa dòng khuyết nhãn Quantity**: Thay vì điền khuyết biến mục tiêu (target) bằng Mean để tránh nhiễu nhân tạo, chúng ta tiến hành xóa bỏ các dòng bị khuyết Quantity ở cả tập Train và Test.
4. **Discount Applied**: Điền bằng `False` (giả định không có chiết khấu).

In [2]:
# 1. Khôi phục Price Per Unit bằng Total Spent / Quantity
def impute_price(row):
    if pd.isnull(row['Price Per Unit']) and not pd.isnull(row['Quantity']) and not pd.isnull(row['Total Spent']):
        return row['Total Spent'] / row['Quantity']
    return row['Price Per Unit']

train_df['Price Per Unit'] = train_df.apply(impute_price, axis=1)
test_df['Price Per Unit'] = test_df.apply(impute_price, axis=1)

print('Số dòng khuyết Price Per Unit sau xử lý:')
print(f'Train: {train_df["Price Per Unit"].isnull().sum()} dòng | Test: {test_df["Price Per Unit"].isnull().sum()} dòng')

Số dòng khuyết Price Per Unit sau xử lý:
Train: 0 dòng | Test: 0 dòng


### Khôi phục cột Item sử dụng bản đồ liên kết (Category, Price Per Unit)

In [3]:
# Tạo lookup dictionary từ tập Train (các dòng không khuyết Item & Price)
lookup_df = train_df.dropna(subset=['Item', 'Price Per Unit'])
item_map = lookup_df.groupby(['Category', 'Price Per Unit'])['Item'].first().to_dict()

# Hàm phục hồi tên Item
def restore_item(row):
    if pd.isnull(row['Item']) and not pd.isnull(row['Price Per Unit']):
        key = (row['Category'], row['Price Per Unit'])
        return item_map.get(key, row['Item'])
    return row['Item']

train_df['Item'] = train_df.apply(restore_item, axis=1)
test_df['Item'] = test_df.apply(restore_item, axis=1)

print('Số dòng khuyết Item sau xử lý:')
print(f'Train: {train_df["Item"].isnull().sum()} dòng | Test: {test_df["Item"].isnull().sum()} dòng')

Số dòng khuyết Item sau xử lý:
Train: 0 dòng | Test: 0 dòng


### Xử lý khuyết thiếu cột Quantity (xóa bỏ) và cột Discount Applied (điền False)

In [4]:
# Xóa bỏ các dòng khuyết biến mục tiêu Quantity để bảo đảm chất lượng huấn luyện
train_df = train_df.dropna(subset=['Quantity'])
test_df = test_df.dropna(subset=['Quantity'])

# Điền khuyết cột Discount Applied bằng False
train_df['Discount Applied'] = train_df['Discount Applied'].fillna(False).astype(bool)
test_df['Discount Applied'] = test_df['Discount Applied'].fillna(False).astype(bool)

print('-'*50)
print('Kiểm tra lại số lượng giá trị khuyết thiếu:')
print('Tập Train:')
print(train_df.isnull().sum())
print('\nTập Test:')
print(test_df.isnull().sum())

--------------------------------------------------
Kiểm tra lại số lượng giá trị khuyết thiếu:
Tập Train:
Transaction ID      0
Customer ID         0
Category            0
Item                0
Price Per Unit      0
Quantity            0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
Discount Applied    0
dtype: int64

Tập Test:
Transaction ID      0
Customer ID         0
Category            0
Item                0
Price Per Unit      0
Quantity            0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
Discount Applied    0
dtype: int64


## Bước 3. Tạo Đặc Trưng Mới (Feature Creation)

Chúng ta tiến hành xây dựng các đặc trưng mới giúp mô hình dự đoán tốt hơn:
1. **Đặc trưng thời gian**: Trích xuất từ cột `Transaction Date` (Year, Month, Day, DayOfWeek, IsWeekend).
2. **Đặc trưng hành vi khách hàng**: Thống kê số giao dịch, trung bình số lượng mua và trung bình chi tiêu của từng `Customer ID` (chỉ tính toán trên tập Train để tránh leakage).
3. **Target Encoding (Mã hóa theo biến mục tiêu)**: Tính toán trung bình `Quantity` của từng loại mặt hàng (`Item`) và danh mục (`Category`) trên tập Train.

In [5]:
# 1. Đặc trưng thời gian từ Transaction Date
for df_temp in [train_df, test_df]:
    dates = pd.to_datetime(df_temp['Transaction Date'])
    df_temp['Txn_Year'] = dates.dt.year
    df_temp['Txn_Month'] = dates.dt.month
    df_temp['Txn_Day'] = dates.dt.day
    df_temp['Txn_DayOfWeek'] = dates.dt.dayofweek
    df_temp['Txn_IsWeekend'] = (dates.dt.dayofweek >= 5).astype(int)

print('Các đặc trưng thời gian được tạo thành công!')

Các đặc trưng thời gian được tạo thành công!

### Tạo đặc trưng hành vi khách hàng (Customer Aggregations)

In [6]:
# Tính toán trên tập Train để tránh rò rỉ dữ liệu
cust_stats = train_df.groupby('Customer ID').agg(
    Customer_Txn_Count=('Transaction ID', 'count'),
    Customer_Avg_Quantity=('Quantity', 'mean'),
    Customer_Avg_Spent=('Total Spent', 'mean')
).reset_index()

# Tính giá trị trung bình toàn cục của tập Train để điền khuyết cho các khách hàng mới ở tập Test (nếu có)
global_txn_count = cust_stats['Customer_Txn_Count'].mean()
global_avg_qty = cust_stats['Customer_Avg_Quantity'].mean()
global_avg_spent = cust_stats['Customer_Avg_Spent'].mean()

# Gộp các đặc trưng hành vi khách hàng vào tập Train và Test
train_df = train_df.merge(cust_stats, on='Customer ID', how='left')
test_df = test_df.merge(cust_stats, on='Customer ID', how='left')

# Điền khuyết cho các khách hàng mới không xuất hiện trong tập Train
for df_temp in [train_df, test_df]:
    df_temp['Customer_Txn_Count'] = df_temp['Customer_Txn_Count'].fillna(global_txn_count)
    df_temp['Customer_Avg_Quantity'] = df_temp['Customer_Avg_Quantity'].fillna(global_avg_qty)
    df_temp['Customer_Avg_Spent'] = df_temp['Customer_Avg_Spent'].fillna(global_avg_spent)

print('Các đặc trưng hành vi khách hàng được tạo thành công!')

Các đặc trưng hành vi khách hàng được tạo thành công!


### Target Encoding (Mã hóa theo biến mục tiêu Quantity)

Tính trung bình `Quantity` của từng loại mặt hàng (`Item`) và từng danh mục (`Category`) trên tập Train và ánh xạ sang cả hai tập.

In [7]:
# Tính toán target encoding trên Train
item_target_enc = train_df.groupby('Item')['Quantity'].mean().to_dict()
cat_target_enc = train_df.groupby('Category')['Quantity'].mean().to_dict()

# Tính giá trị trung bình toàn cục trên Train để điền khuyết cho các mặt hàng mới ở tập Test (nếu có)
global_qty_mean = train_df['Quantity'].mean()

# Ánh xạ đặc trưng mã hóa vào Train và Test
train_df['Item_Target_Enc'] = train_df['Item'].map(item_target_enc).fillna(global_qty_mean)
test_df['Item_Target_Enc'] = test_df['Item'].map(item_target_enc).fillna(global_qty_mean)

train_df['Category_Target_Enc'] = train_df['Category'].map(cat_target_enc).fillna(global_qty_mean)
test_df['Category_Target_Enc'] = test_df['Category'].map(cat_target_enc).fillna(global_qty_mean)

print('Đặc trưng Target Encoding đã được xây dựng thành công!')

Đặc trưng Target Encoding đã được xây dựng thành công!


## Bước 4. Mã Hóa Biến Phân Loại & Chuẩn Hóa Biến Số (Encoding & Scaling)

1. **One-Hot Encoding**: Áp dụng cho `Payment Method` và `Location`.
2. **Label Encoding / Binary**: Đưa cột `Discount Applied` về dạng `0` và `1`.
3. **Standard Scaling**: Chuẩn hóa **tất cả** các cột số liên tục (bao gồm cả Price Per Unit và Total Spent) để đảm bảo SVM hội tụ.

In [8]:
# 1. Đưa cột Discount Applied về dạng số 0 và 1
train_df['Discount Applied'] = train_df['Discount Applied'].astype(int)
test_df['Discount Applied'] = test_df['Discount Applied'].astype(int)

# 2. One-Hot Encoding cho Payment Method và Location
train_df = pd.get_dummies(train_df, columns=['Payment Method', 'Location'], drop_first=True, dtype=int)
test_df = pd.get_dummies(test_df, columns=['Payment Method', 'Location'], drop_first=True, dtype=int)

# Đảm bảo tập Test có cùng số cột và định dạng với tập Train
test_df = test_df.reindex(columns=train_df.columns, fill_value=0)

print('Đã thực hiện mã hóa các biến phân loại!')

Đã thực hiện mã hóa các biến phân loại!


### Chuẩn hóa biến số bằng StandardScaler

Để mô hình SVM không bị phân kỳ, chúng ta **chuẩn hóa toàn bộ** các cột đặc trưng dạng số bao gồm cả các cột ngày tháng, giá và tổng chi tiêu.

In [9]:
num_cols_to_scale = [
    'Price Per Unit', 'Total Spent', 'Txn_Year', 'Txn_Month', 'Txn_Day', 'Txn_DayOfWeek', 'Txn_IsWeekend',
    'Customer_Txn_Count', 'Customer_Avg_Quantity', 'Customer_Avg_Spent',
    'Item_Target_Enc', 'Category_Target_Enc'
]

scaler = StandardScaler()

# Fit scaler trên tập Train
train_df[num_cols_to_scale] = scaler.fit_transform(train_df[num_cols_to_scale])

# Transform trên tập Test
test_df[num_cols_to_scale] = scaler.transform(test_df[num_cols_to_scale])

print('Đã hoàn thành chuẩn hóa Standard Scaling cho tất cả các đặc trưng!')

Đã hoàn thành chuẩn hóa Standard Scaling cho tất cả các đặc trưng!


## Bước 5. Lựa Chọn Đặc Trưng (Feature Selection)

Loại bỏ các cột định danh dạng chuỗi không mang giá trị học máy. Chúng ta lưu lại cả `Price Per Unit` và `Total Spent` ra file để linh hoạt lựa chọn đặc trưng trong phần Modeling.

In [10]:
cols_to_drop = ['Transaction ID', 'Customer ID', 'Category', 'Item', 'Transaction Date']

X_train = train_df.drop(columns=cols_to_drop + ['Quantity'])
y_train = train_df['Quantity']

X_test = test_df.drop(columns=cols_to_drop + ['Quantity'])
y_test = test_df['Quantity']

print('Các cột thuộc tính đưa vào mô hình:')
print(list(X_train.columns))
print('-'*50)
print(f'Kích thước X_train: {X_train.shape} | y_train: {y_train.shape}')
print(f'Kích thước X_test: {X_test.shape} | y_test: {y_test.shape}')

Các cột thuộc tính đưa vào mô hình:
['Price Per Unit', 'Total Spent', 'Discount Applied', 'Txn_Year', 'Txn_Month', 'Txn_Day', 'Txn_DayOfWeek', 'Txn_IsWeekend', 'Customer_Txn_Count', 'Customer_Avg_Quantity', 'Customer_Avg_Spent', 'Item_Target_Enc', 'Category_Target_Enc', 'Payment Method_Credit Card', 'Payment Method_Digital Wallet', 'Location_Online']
--------------------------------------------------
Kích thước X_train: (9577, 16) | y_train: (9577,)
Kích thước X_test: (2394, 16) | y_test: (2394,)


## Bước 6. Lưu Dữ Liệu Đã Xử Lý (Save Cleaned Datasets)

Lưu các tập dữ liệu huấn luyện và kiểm thử đã làm sạch và xử lý ra các file CSV tương ứng trong thư mục `ready_train`.

In [11]:
output_dir = '../data/ready_train/'
os.makedirs(output_dir, exist_ok=True)

# Tạo dataframe hoàn chỉnh để lưu (bao gồm cả biến mục tiêu)
train_processed = X_train.copy()
train_processed['Quantity'] = y_train

test_processed = X_test.copy()
test_processed['Quantity'] = y_test

train_processed.to_csv(os.path.join(output_dir, 'train_ready.csv'), index=False)
test_processed.to_csv(os.path.join(output_dir, 'test_ready.csv'), index=False)

print(f'Đã lưu train_ready.csv thành công tại: {os.path.abspath(os.path.join(output_dir, "train_ready.csv"))}')
print(f'Đã lưu test_ready.csv thành công tại: {os.path.abspath(os.path.join(output_dir, "test_ready.csv"))}')

Đã lưu train_ready.csv thành công tại: C:\Users\Win 11\Documents\ML\ML_proj_26\lab2\data\ready_train\train_ready.csv
Đã lưu test_ready.csv thành công tại: C:\Users\Win 11\Documents\ML\ML_proj_26\lab2\data\ready_train\test_ready.csv
